# Monte Carlo & Synthetic-Market Robustness Suite
**Strategies:** `bitcoin_crash_strategy.py` (BTC) and `eth_final_strategy.py` (ETH), embedded below and run exactly as written with their frozen parameters, 0.15 % cost per side and 5 % short carry.

| Part | Test | Question it answers |
|---|---|---|
| **A** | Monte Carlo: **trade resampling** | How much of the historical result is luck in *which trades happened* and *in what order*? |
| **B** | Monte Carlo: **parameter perturbation** | Does the edge survive if every parameter is a bit "wrong"? (overfitting check) |
| **C** | **Synthetic market test**: `trend_bull`, `momentum_mild`, `bubble_burst`, `chop_low_vol` | How does the strategy behave in market regimes it was not tuned on? |

**How to use:** put this notebook in the same folder as `BTCUSDT.csv` and `ETHUSDT.csv` (15-minute OHLCV) and run the cells top to bottom. The two strategy files are written to disk by Section 2. All knobs are in the Settings cell.

## 1 · Setup

In [ ]:
import importlib.util, subprocess, sys, time, warnings
from dataclasses import replace
from pathlib import Path

try:
    import numba
except ImportError:
    print("numba not found -> installing ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "numba"])
    import numba

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import lfilter
from scipy.stats import spearmanr
from IPython.display import display

warnings.filterwarnings("ignore")

DATA_DIR       = Path(".")
ASSETS         = ["BTC", "ETH"]
SEED           = 26
INITIAL        = 100_000.0

N_TRADE_SIMS   = 5000
BLOCK          = 5
SKIP_FRAC      = 0.15

N_PARAM_RUNS   = 150
PERTURB_PCT    = 0.20

N_SYNTH_PATHS  = 20
SYNTH_DAYS     = 1095

OUT_DIR = Path("mc_results"); OUT_DIR.mkdir(exist_ok=True)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": .25,
                     "axes.spines.top": False, "axes.spines.right": False, "font.size": 9})
C_STRAT, C_BH, C_BASE, C_BAD = "#1f77b4", "#7f7f7f", "#ff7f0e", "#d62728"
print("Settings loaded. numba", numba.__version__)

## 2 · Strategy modules
### `bitcoin_crash_strategy.py`

In [ ]:
%%writefile bitcoin_crash_strategy.py
from dataclasses import dataclass, asdict, replace
from pathlib import Path
import argparse, json
import numpy as np
import pandas as pd
try:
    from numba import njit
except ImportError:
    def njit(*a, **k): return lambda f:f
INITIAL=100000.

@dataclass(frozen=True)
class Config:
    rule: str='4h'
    offset_hours: int=0
    trend_mode: int=0
    momentum_days: int=40
    entry_threshold: float=.66
    exit_threshold: float=0.
    vol_fast_days: int=7
    vol_slow_days: int=30
    regime_days: int=75
    breakout: int=30
    exit_bars: int=9
    stop_atr: float=2.5
    lock_trigger: float=1.5
    lock_profit: float=.5
    trail_atr: float=0.
    vol_rank_max: float=.8
    vol_target: float=1.2
    weekends: bool=False
    long_cap: float=1.
    short_hysteresis: float=0.
    short_require_long_bear: bool=False
    short_mode: int=1
    short_cap: float=.65
    short_breakout: int=24
    short_exit: int=6
    short_stop: float=2.5
    short_lock: float=1.5
    short_vol_rank: float=1.01
    short_regime_days: int=0
    short_slope_days: int=5
    short_regime_exit: bool=True
    short_trail_atr: float=0.
    short_lock_profit: float=-1.
    short_vol_target: float=0.
    short_signal_mode: int=0
    short_rally_ema: int=20
    short_shock_atr: float=0.
    short_shock_bars: int=6
    only_shorts: bool=False
    regime_exit: bool=False
    hurst_min: float=0.
    long_adx_min: float=0.
    short_adx_min: float=0.
    short_hurst_min: float=0.
    channel_close: bool=False
    pullback_rsi: float=0.
    dd_soft: float=.99
    dd_hard: float=.999
    cooldown_bars: int=4
    max_hold_bars: int=5760
    cost: float=.0015
    carry: float=.05
    delay: int=0
    long_short: bool=True

class Features:
    def __init__(self,d):
        self.d=d;self.cache={};self.day=resample(d,'1D')
    def get(self,p):
        per=int(pd.Timedelta('1D')/pd.Timedelta(p.rule))
        key=(p.rule,p.offset_hours,p.vol_fast_days,p.vol_slow_days)
        if key not in self.cache:
            b=resample(self.d,p.rule,pd.Timedelta(hours=p.offset_hours));per=int(pd.Timedelta('1D')/pd.Timedelta(p.rule))
            lr=np.log(b.close).diff()
            v=lr.rolling(p.vol_slow_days*per).std()*np.sqrt(365*per)
            b['vol']=pd.concat([lr.rolling(p.vol_fast_days*per).std()*np.sqrt(365*per),v],axis=1).max(axis=1)
            b['rank']=v.rolling(90*per,min_periods=30*per).rank(pct=True)

            logp=np.log(b.close)
            v1=logp.diff().rolling(90*per,min_periods=30*per).var()
            v16=logp.diff(16).rolling(90*per,min_periods=30*per).var()
            b['hurst']=(np.log(v16/v1)/(2*np.log(16))).clip(0,1)
            up=b.high.diff();dn=-b.low.diff();plus=up.where((up>dn)&(up>0),0.).ewm(alpha=1/14,adjust=False).mean();minus=dn.where((dn>up)&(dn>0),0.).ewm(alpha=1/14,adjust=False).mean()
            b['adx']=(100*(plus-minus).abs()/(plus+minus)).ewm(alpha=1/14,adjust=False,min_periods=28).mean()
            self.cache[key]=b
        b=self.cache[key];ma=self.day.close.rolling(p.regime_days).mean()
        bull=((self.day.close>ma)&self.day.complete).reindex(b.index,method='ffill',fill_value=False)
        sma=self.day.close.rolling(p.short_regime_days or p.regime_days).mean()
        bear=((self.day.close<sma*(1-p.short_hysteresis))&(sma<sma.shift(p.short_slope_days))&self.day.complete).reindex(b.index,method='ffill',fill_value=False)
        if p.short_shock_atr>0:
            fast=b.close.ewm(span=12,adjust=False).mean()
            shock=(b.close-b.close.shift(p.short_shock_bars)<-p.short_shock_atr*b.atr.shift(p.short_shock_bars))&(b.close<fast)&(fast<fast.shift(3))
            bear= bear|shock
        en=((b.close>(b.close if p.channel_close else b.high).shift().rolling(p.breakout).max())|((b.rsi<p.pullback_rsi)&(p.pullback_rsi>0)))&bull&(b['rank']<=p.vol_rank_max)&(b.hurst>=p.hurst_min)&(b.adx>=p.long_adx_min)
        if not p.weekends:en&=b.index.dayofweek<5
        sn=(b.close<b.low.shift().rolling(p.short_breakout).min())&bear&(b['rank']<=p.short_vol_rank)&(b.hurst>=p.short_hurst_min)&(b.adx>=p.short_adx_min)
        if p.short_signal_mode==1:sn=(b.close<b.close.ewm(span=p.short_breakout,adjust=False).mean())&(b.close<b.close.shift(3))&bear&(b['rank']<=p.short_vol_rank)&(b.hurst>=p.short_hurst_min)&(b.adx>=p.short_adx_min)
        if p.short_signal_mode in (2,3):
            rally_ema=b.close.ewm(span=p.short_rally_ema,adjust=False).mean()
            failed_rally=(b.high>=rally_ema.shift())&(b.close<rally_ema)&(b.close<b.open)
            cross=(b.close<rally_ema)&(b.close.shift()>=rally_ema.shift())
            sn=(failed_rally if p.short_signal_mode==2 else cross)&bear&(b['rank']<=p.short_vol_rank)&(b.hurst>=p.short_hurst_min)&(b.adx>=p.short_adx_min)
        if p.short_mode==2:sn&=b.close<b.close.ewm(span=20,adjust=False).mean()
        if p.short_require_long_bear:sn &= ~bull
        if p.short_mode==0 or not p.long_short:sn&=False
        lx=b.close<b.low.shift().rolling(p.exit_bars).min()
        sx=(b.close>b.high.shift().rolling(p.short_exit).max())|(~bear & p.short_regime_exit)
        if p.regime_exit:lx|=~bull
        if p.trend_mode:
            score=pd.concat([np.sign(b.close/b.close.shift(int(p.momentum_days*k*per))-1) for k in (.5,1.,2.)],axis=1).mean(axis=1)
            en=(score>=p.entry_threshold)&bull&(b['rank']<=p.vol_rank_max)&(b.hurst>=p.hurst_min)&(b.adx>=p.long_adx_min)
            sn=(score<=-p.entry_threshold)&bear&(b.hurst>=p.short_hurst_min)&(b.adx>=p.short_adx_min)&(p.long_short)&(p.short_mode!=0)
            lx=score<=p.exit_threshold; sx=(score>=-p.exit_threshold)|~bear
            if p.trend_mode==2: en=bull;sn=bear&(p.long_short);lx=~bull;sx=~bear
            if not p.weekends: en &= b.index.dayofweek<5
        if p.only_shorts:en &= False
        frame=pd.DataFrame(dict(signal_close=b.close,en=en,sn=sn,lx=lx,sx=sx,valid=b.complete,atr=b.atr,size=pd.Series(np.where(en,p.vol_target,p.short_vol_target or p.vol_target),index=b.index).div(b.vol).clip(upper=1),hurst=b.hurst,vol=b.vol,vol_rank=b['rank'],adx=b.adx,bull=bull,bear=bear,short_break=(b.close<b.low.shift().rolling(p.short_breakout).min()),short_below_ma=(self.day.close<sma).reindex(b.index,method='ffill',fill_value=False),short_slope=(sma<sma.shift(p.short_slope_days)).reindex(b.index,method='ffill',fill_value=False)))
        aligned=frame.reindex(self.d.index)
        if p.delay:aligned=aligned.shift(p.delay)

        self.update_clock=aligned.valid.eq(True).to_numpy(bool)
        self.update_prices=aligned.signal_close.fillna(0).to_numpy(float)


        events=np.flatnonzero(aligned.valid.eq(True).to_numpy())
        eligible=np.flatnonzero(self.d.tradable.to_numpy())
        destinations=np.searchsorted(eligible,events)
        keep=destinations<len(eligible)
        events=events[keep]; destinations=eligible[destinations[keep]]

        unique=~pd.Index(destinations).duplicated(keep='last')
        queued=aligned.iloc[events[unique]].copy();queued.index=self.d.index[destinations[unique]]
        aligned=queued.reindex(self.d.index)
        flags=[aligned[k].eq(True).to_numpy(bool) for k in ['en','sn','lx','sx','valid']]
        vals=[pd.to_numeric(aligned[k]).fillna(0).to_numpy(float) for k in ['atr','size']]
        return (*flags,*vals),frame

@njit(cache=True)
def simulate(o,h,l,c,liquid,en,sn,lx,sx,valid,atr,size,start,end,initial,cost,carry,stop_atr,lock_trigger,lock_profit,trail_atr,short_cap,short_stop,short_lock,ddsoft,ddhard,cooldown,maxhold,update_clock=None,short_trail_atr=0.,short_lock_profit=-1.,long_cap=1.,update_prices=None):
    n=len(c);eq=np.full(n,np.nan);exp=np.zeros(n);cp=np.full(n,np.nan);qp=np.zeros(n);dd=np.zeros(n);cy=np.zeros(n)
    stop_path=np.full(n,np.nan)
    orders=np.zeros((2*n+4,8));trades=np.zeros((n+2,7));oc=tc=0
    cash=initial;q=0.;stop=entry=ea=0.;peak=initial;born=start;until=start;ep_base=initial;fees=ecarry=totalcarry=0.;locked=False
    maxgross=0.;gaps=intrabr=0;recovering=False;risk_pending=False
    last=end-1

    for i in range(start,end):
        if i>start and q<0:
            charge=abs(q)*c[i-1]*carry/35040.;cash-=charge;totalcarry+=charge;ecarry+=charge
        pre=cash+q*o[i];gross=abs(q)*o[i]/max(pre,1e-12);maxgross=max(maxgross,gross)
        if gross>1+1e-9:gaps+=1

        localdd=1-pre/max(peak,1e-12)
        if localdd<=ddsoft:recovering=False
        risktrip=localdd>ddhard and not recovering
        if risktrip:recovering=True;risk_pending=q!=0;until=max(until,i+288)
        reason=0;close=False
        if q!=0 and liquid[i]:
            if (q>0 and o[i]<=stop) or (q<0 and o[i]>=stop):close=True;reason=1
            elif valid[i] and ((q>0 and lx[i]) or (q<0 and sx[i])):close=True;reason=0
            elif valid[i] and ((q>0 and sn[i]) or (q<0 and en[i])):close=True;reason=8
            elif i-born>=maxhold:close=True;reason=4
            elif risk_pending:close=True;reason=5
            elif gross>=.995 and q<0:close=True;reason=6
        if i==last and q!=0 and liquid[i]:close=True;reason=7
        if close:
            risk_pending=False
            side=np.sign(q);delta=-q;fee=abs(q)*o[i]*cost;cash-=delta*o[i]+fee;fees+=fee
            orders[oc]=np.array([i,delta,o[i],fee,reason,q,0.,cash]);oc+=1
            trades[tc]=np.array([born,i,side,cash-ep_base,fees,ecarry,(i-born)/96.]);tc+=1;q=0.
            if reason in (1,4,5,6):until=max(until,i+cooldown)
        if q==0 and i<last and i>=until and valid[i] and liquid[i] and atr[i]>0:
            side=1. if en[i] else (-1. if sn[i] else 0.)
            if side!=0:
                fraction=min(long_cap,size[i]) if side>0 else min(short_cap,size[i])
                localdd=1-cash/max(peak,1e-12)
                if localdd>ddsoft:fraction*=.5
                if localdd>ddhard:fraction*=.5
                ep_base=cash;fees=ecarry=0.;born=i;entry=o[i];ea=atr[i];locked=False
                q=side*fraction*cash/(o[i]*(1+fraction*cost));fee=abs(q)*o[i]*cost;cash-=q*o[i]+fee;fees+=fee
                stop=entry-side*(stop_atr if side>0 else short_stop)*ea
                orders[oc]=np.array([i,q,o[i],fee,9,0.,q,cash]);oc+=1
        if q!=0:
            side=np.sign(q);price=stop;reason=2
            if q<0:
                capprice=cash/(abs(q)*(2+cost))*.995
                if capprice<price:price=capprice;reason=6
            trigger=liquid[i] and ((q>0 and l[i]<=price) or (q<0 and h[i]>=price))
            if trigger:price=min(o[i],price) if q>0 else max(o[i],price)

            adverse=h[i] if q>0 or not trigger else price
            wg=abs(q)*adverse/max(cash+q*adverse,1e-12);maxgross=max(maxgross,wg)
            if wg>1+1e-9:intrabr+=1
            if trigger:
                delta=-q;fee=abs(q)*price*cost;cash-=delta*price+fee;fees+=fee
                orders[oc]=np.array([i,delta,price,fee,reason,q,0.,cash]);oc+=1
                trades[tc]=np.array([born,i,side,cash-ep_base,fees,ecarry,(i-born)/96.]);tc+=1
                q=0.;until=i+cooldown
            elif i+1<end and (valid[i+1] if update_clock is None else update_clock[i+1]):
                completed_close=c[i] if update_prices is None else update_prices[i+1]
                profit=side*(completed_close-entry);threshold=lock_trigger if side>0 else short_lock
                if not locked and profit>=threshold*ea:
                    locked=True;floor=short_lock_profit if side<0 and short_lock_profit>=0 else lock_profit
                    level=entry+side*max(floor*ea,entry*2.2*cost)
                    stop=max(stop,level) if side>0 else min(stop,level)
                trailing=short_trail_atr if side<0 else trail_atr
                if trailing>0 and locked:
                    level=completed_close-side*trailing*ea;stop=max(stop,level) if side>0 else min(stop,level)
        stop_path[i]=stop if q!=0 else np.nan
        equity=cash+q*c[i];peak=max(peak,equity);eq[i]=equity;exp[i]=abs(q)*c[i]/max(equity,1e-12);cp[i]=cash;qp[i]=q;dd[i]=1-equity/peak;cy[i]=totalcarry
    return eq,exp,cp,qp,dd,cy,orders[:oc],trades[:tc],maxgross,gaps,intrabr,stop_path

def run(d,p,features=None,start=None,end=None,initial=INITIAL):

    if not (0<=p.short_cap<1 and 0<p.long_cap<=1 and p.cost>=0 and p.carry>=0 and p.vol_target>0):raise ValueError('Invalid configuration')
    f=features or Features(d);args,diagnostics=f.get(p)
    a=0 if start is None else int(d.index.searchsorted(pd.Timestamp(start,tz='UTC')))
    z=len(d) if end is None else int(d.index.searchsorted(pd.Timestamp(end,tz='UTC')))
    if z-a<2:raise ValueError('Empty window')
    x=simulate(*(d[k].to_numpy() for k in ['open','high','low','close','tradable']),*args,a,z,initial,p.cost,p.carry,p.stop_atr,p.lock_trigger,p.lock_profit,p.trail_atr,p.short_cap,p.short_stop,p.short_lock,p.dd_soft,p.dd_hard,p.cooldown_bars,p.max_hold_bars,f.update_clock,p.short_trail_atr,p.short_lock_profit,p.long_cap,f.update_prices)
    path=pd.DataFrame(dict(equity=x[0],exposure=x[1],cash=x[2],units=x[3],local_drawdown=x[4],carry=x[5]),index=d.index).iloc[a:z]
    path['standing_stop_next_bar']=x[11][a:z]
    path['signed_exposure']=path.units*d.close.reindex(path.index)/path.equity
    fills=pd.DataFrame(x[6],columns=['bar','delta_units','price','cost','reason','closed_units','result_units','cash_after']);fills['timestamp']=d.index[fills.bar.astype(int)]
    tr=pd.DataFrame(x[7],columns=['entry_bar','exit_bar','direction','pnl','cost','carry','days']);tr['entry_time']=d.index[tr.entry_bar.astype(int)];tr['exit_time']=d.index[tr.exit_bar.astype(int)]
    if abs(path.units.iloc[-1])<1e-8 and abs(path.equity.iloc[-1]-initial-tr.pnl.sum())>1e-5:raise AssertionError('Episode ledger mismatch')
    path['restricted_collateral']=2*abs(path.units.clip(upper=0))*d.close.reindex(path.index)
    path['free_cash']=path.cash-path.restricted_collateral
    audit=dict(max_observed_gross=float(x[8]),gap_cap_breaches=int(x[9]),intrabar_cap_breaches=int(x[10]),all_fills_tradable=bool(d.tradable.reindex(fills.timestamp).all()),terminal_flat=bool(abs(path.units.iloc[-1])<1e-8))
    return path,fills,tr,audit

def load_data(file):

    raw = pd.read_csv(file)
    tc = next((x for x in ('timestamp','open_time','time','datetime','date') if x in raw),None)
    if tc is None: raise ValueError('Missing timestamp column')
    idx = pd.to_datetime(raw[tc].astype(str),format='ISO8601',utc=True)
    if idx.duplicated().any(): raise ValueError('Duplicate timestamps')
    d = raw[['open','high','low','close','volume']].apply(pd.to_numeric,errors='raise')
    d.index = idx
    if not np.isfinite(d.to_numpy()).all(): raise ValueError('Nonfinite OHLCV')
    if (d[['open','high','low','close']]<=0).any().any() or (d.volume<0).any(): raise ValueError('Invalid prices/volume')
    if ((d.high<d[['open','close','low']].max(axis=1))|(d.low>d[['open','close','high']].min(axis=1))).any(): raise ValueError('OHLC envelope violation')
    filled = pd.Series(False,index=idx)
    if 'is_filled' in raw:
        flags = raw.is_filled.astype(str).str.lower()
        if not flags.isin(['true','false','0','1']).all(): raise ValueError('Invalid is_filled flag')
        filled = pd.Series(flags.isin(['true','1']).to_numpy(),index=idx)
    d['tradable'] = (d.volume>0)&~filled
    d = d.sort_index()
    offgrid = d.index != d.index.floor('15min')
    audit = dict(raw_bars=len(d),off_grid_excluded=int(offgrid.sum()),zero_volume=int((d.volume==0).sum()),filled_flag=int(filled.sum()))
    if offgrid.any(): raise ValueError('Off-grid timestamps: refusing silent exclusions')
    d = d.loc[~offgrid]
    if d.index.to_series().diff().dropna().median()!=pd.Timedelta('15min'): raise ValueError('Expected 15-minute data; refusing coarse-data upsampling')
    if not len(d): raise ValueError('No 15-minute bars')
    grid = pd.date_range(d.index[0],d.index[-1],freq='15min')
    d = d.reindex(grid)
    missing = d.close.isna()
    previous = d.close.ffill()
    for c in ('open','high','low','close'): d[c] = d[c].where(~missing,previous)
    d['volume'] = d.volume.fillna(0)
    d['tradable'] = d.tradable.eq(True)
    d.index.name = 'timestamp'
    audit.update(grid_bars=len(d),missing_synthetic=int(missing.sum()),untradable=int((~d.tradable).sum()),start=str(d.index[0]),end=str(d.index[-1]))
    d.attrs['audit'] = audit
    return d

def resample(d,rule,offset=None):
    b = d.resample(rule,origin='start_day',offset=offset).agg({'open':'first','high':'max','low':'min','close':'last','volume':'sum','tradable':'sum'})
    b.index += pd.Timedelta(rule)
    expected = int(pd.Timedelta(rule)/pd.Timedelta('15min'))
    b['complete'] = b.tradable==expected
    prev=b.close.shift()
    tr=pd.concat([b.high-b.low,(b.high-prev).abs(),(b.low-prev).abs()],axis=1).max(axis=1)
    b['atr']=tr.ewm(alpha=1/14,adjust=False,min_periods=14).mean()
    r=b.close.diff();up=r.clip(lower=0).ewm(alpha=1/3,adjust=False,min_periods=3).mean();dn=(-r.clip(upper=0)).ewm(alpha=1/3,adjust=False,min_periods=3).mean()
    b['rsi']=(100*up/(up+dn)).fillna(50)
    return b

def benchmark(d,cost=.0015,initial=INITIAL):
    ix=np.flatnonzero(d.tradable.to_numpy())
    if not len(ix) or not d.tradable.iloc[-1]: raise ValueError('Benchmark terminal open unavailable; no retrospective terminal fill')
    a,z=ix[0],len(d)-1
    qty=initial/(d.open.iloc[a]*(1+cost));out=pd.Series(initial,index=d.index,dtype=float)
    out.iloc[a:z+1]=qty*d.close.iloc[a:z+1]
    out.iloc[z:]=qty*d.open.iloc[z]*(1-cost)
    return out

def quarters(eq,bh,initial=INITIAL):
    rows=[];prev=prevbh=initial
    for period,s in eq.groupby(eq.index.tz_localize(None).to_period('Q')):
        b=bh.reindex(s.index);complete=s.index[0]<=period.start_time.tz_localize('UTC') and s.index[-1]>=period.end_time.floor('15min').tz_localize('UTC')
        sr=s.iloc[-1]/prev-1;br=b.iloc[-1]/prevbh-1
        rows.append(dict(quarter=str(period),strategy_return_pct=100*sr,buy_hold_return_pct=100*br,excess_pct=100*(sr-br),beat=bool(sr>br),complete=bool(complete)))
        prev=s.iloc[-1];prevbh=b.iloc[-1]
    return pd.DataFrame(rows)

def metrics(path,fills,tr,bh,audit,initial=INITIAL):

    eq=path.equity;days=(len(eq)*15/1440);daily=eq.resample('1D').last();r=daily.pct_change();r.iloc[0]=daily.iloc[0]/initial-1;r=r.fillna(0)
    downside=np.sqrt(np.mean(np.minimum(r,0)**2));sh=np.sqrt(365)*r.mean()/r.std(ddof=1) if r.std()>0 else 0.
    dd=eq/pd.Series(np.maximum.accumulate(np.r_[initial,eq.to_numpy()])[1:],index=eq.index)-1
    under=dd<-1e-10;longest=cur=0
    for u in under:cur=cur+1 if u else 0;longest=max(longest,cur)
    q=quarters(eq,bh,initial);qc=q[q.complete];wins=tr.loc[tr.pnl>0,'pnl'];loss=tr.loc[tr.pnl<=0,'pnl']
    m=dict(total_return_pct=float(100*(eq.iloc[-1]/initial-1)),short_trades=int((tr.direction<0).sum()),same_open_reversals=int(sum(len(g)==2 and g.iloc[0].closed_units*g.iloc[1].result_units<0 for _,g in fills.groupby('bar'))),cagr_pct=100*((eq.iloc[-1]/initial)**(365.25/days)-1),net_profit=float(eq.iloc[-1]-initial),final_equity=float(eq.iloc[-1]),sharpe=float(sh),sortino=float(np.sqrt(365)*r.mean()/downside) if downside>0 else 0.,max_drawdown_pct=float(-100*dd.min()),mean_exposure_pct=float(100*path.exposure.mean()),time_in_market_pct=float(100*(path.exposure>1e-6).mean()),longest_underwater_days=longest/96.,cost_paid=float(fills.cost.sum()),carry_paid=float(path.carry.iloc[-1]),executed_orders=len(fills),total_closed_trades=len(tr),win_rate_pct=float(100*(tr.pnl>0).mean()) if len(tr) else 0.,gross_profit=float(wins.sum()),gross_loss=float(-loss.sum()),average_winning_trade=float(wins.mean()) if len(wins) else 0.,average_losing_trade=float(loss.mean()) if len(loss) else 0.,largest_winning_trade=float(wins.max()) if len(wins) else 0.,largest_losing_trade=float(loss.min()) if len(loss) else 0.,average_holding_days=float(tr.days.mean()) if len(tr) else 0.,maximum_holding_days=float(tr.days.max()) if len(tr) else 0.,buy_hold_return_pct=float(100*(bh.iloc[-1]/initial-1)),quarter_beats=int(qc.beat.sum()),complete_quarters=len(qc),quarter_beat_pct=float(100*qc.beat.mean()) if len(qc) else 0.,daily_observations=len(r),**audit)
    gp=tr.pnl+tr.cost+tr.carry
    m.update(gross_profit_before_cost=float(gp.clip(lower=0).sum()),gross_loss_before_cost=float(-gp.clip(upper=0).sum()),pnl_before_cost_total=float(gp.sum()),annual_realized_vol_pct=float(r.std(ddof=1)*np.sqrt(365)*100),one_way_turnover_equity_pa=float((fills.delta_units.abs()*fills.price).sum()/eq.mean()/(days/365.25)),cost_pct_average_equity_pa=float(100*fills.cost.sum()/eq.mean()/(days/365.25)),short_episode_net_pnl=float(tr.loc[tr.direction<0,'pnl'].sum()),long_episode_net_pnl=float(tr.loc[tr.direction>0,'pnl'].sum()))
    return m,q,r

def export(dest,d,p,path,fills,tr,audit,label,initial=INITIAL):
    dest=Path(dest);dest.mkdir(parents=True,exist_ok=True);bh=benchmark(d.reindex(path.index),p.cost,initial)
    m,q,r=metrics(path,fills,tr,bh,audit,initial);m['result_type']=label
    path.assign(buy_hold=bh).to_csv(dest/'equity.csv');fills.to_csv(dest/'fills.csv',index=False);tr.to_csv(dest/'trades.csv',index=False);q.to_csv(dest/'quarters.csv',index=False)
    (dest/'metrics.json').write_text(json.dumps(m,indent=2));return m


FROZEN_PARAMETERS = {'rule': '4h', 'offset_hours': 0, 'trend_mode': 0, 'momentum_days': 40, 'entry_threshold': 0.66, 'exit_threshold': 0.0, 'vol_fast_days': 7, 'vol_slow_days': 30, 'regime_days': 40, 'breakout': 36, 'exit_bars': 54, 'stop_atr': 4.0, 'lock_trigger': 1.25, 'lock_profit': 1.0, 'trail_atr': 0.0, 'vol_rank_max': 0.9, 'vol_target': 0.8, 'weekends': False, 'long_cap': 1.0, 'short_hysteresis': 0.01, 'short_require_long_bear': True, 'short_mode': 1, 'short_cap': 0.8, 'short_breakout': 72, 'short_exit': 18, 'short_stop': 3, 'short_lock': 1.5, 'short_vol_rank': 1.01, 'short_regime_days': 75, 'short_slope_days': 30, 'short_regime_exit': True, 'short_trail_atr': 4.0, 'short_lock_profit': 1.0, 'short_vol_target': 0.65, 'short_signal_mode': 0, 'short_rally_ema': 20, 'short_shock_atr': 0.0, 'short_shock_bars': 6, 'only_shorts': False, 'regime_exit': False, 'hurst_min': 0.0, 'long_adx_min': 0.0, 'short_adx_min': 0, 'short_hurst_min': 0.45, 'channel_close': False, 'pullback_rsi': 0.0, 'dd_soft': 0.15, 'dd_hard': 0.26, 'cooldown_bars': 96, 'max_hold_bars': 5760, 'cost': 0.0015, 'carry': 0.05, 'delay': 0, 'long_short': True}

def main():
    ap=argparse.ArgumentParser(description=__doc__)
    ap.add_argument('--data',required=True);ap.add_argument('--config');ap.add_argument('--start');ap.add_argument('--end')
    ap.add_argument('--out',default='bitcoin_crash_results');ap.add_argument('--long-only',action='store_true')
    ap.add_argument('--short-only',action='store_true')
    ap.add_argument('--cost',type=float);ap.add_argument('--carry',type=float);ap.add_argument('--delay',type=int)
    args=ap.parse_args();p=Config(**(json.loads(Path(args.config).read_text())['parameters'] if args.config else FROZEN_PARAMETERS))
    for k in ['cost','carry','delay']:
        if getattr(args,k) is not None:p=replace(p,**{k:getattr(args,k)})
    if args.long_only and args.short_only:ap.error('Choose only one directional mode')
    if args.long_only:p=replace(p,long_short=False,only_shorts=False)
    if args.short_only:p=replace(p,long_short=True,only_shorts=True)
    if p.cost<.0015 or p.delay<0 or p.carry<0:ap.error('Cost must be >=0.15% per side; delay and carry nonnegative')
    np.random.seed(26);d=load_data(args.data);x=run(d,p,start=args.start,end=args.end);path,fills,tr,audit=x
    out=Path(args.out);out.mkdir(parents=True,exist_ok=True)
    if audit['gap_cap_breaches'] or audit['intrabar_cap_breaches'] or (path.equity<=0).any() or not audit['terminal_flat']:
        (out/'rejection_audit.json').write_text(json.dumps(audit,indent=2));path.to_csv(out/'rejected_equity.csv')
        raise SystemExit('Replay rejected: observed >1x gross exposure or nonpositive equity. No clipping of losses.')
    m=export(out,d,p,*x,'historical replay; parameters selected using supplied history')
    (out/'used_config.json').write_text(json.dumps(asdict(p),indent=2));(out/'data_audit.json').write_text(json.dumps(d.attrs['audit'],indent=2));print(json.dumps(m,indent=2))
if __name__=='__main__':main()

### `eth_final_strategy.py`

In [ ]:
%%writefile eth_final_strategy.py
from dataclasses import dataclass, asdict, replace
from pathlib import Path
import argparse, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
try:
    from numba import njit
except ImportError:
    def njit(*a, **k): return lambda f:f
INITIAL=100000.

@dataclass(frozen=True)
class Config:
    rule: str='4h'
    offset_hours: int=0
    trend_mode: int=0
    momentum_days: int=40
    entry_threshold: float=.66
    exit_threshold: float=0.
    vol_fast_days: int=7
    vol_slow_days: int=30
    regime_days: int=35
    breakout: int=10
    exit_bars: int=36
    stop_atr: float=2.
    lock_trigger: float=2.5
    lock_profit: float=.5
    trail_atr: float=6.
    vol_rank_max: float=.9
    vol_target: float=1.2
    weekends: bool=True
    short_mode: int=1
    short_cap: float=.65
    short_breakout: int=36
    short_exit: int=18
    short_stop: float=2.5
    short_lock: float=1.5
    short_vol_rank: float=1.01
    short_regime_days: int=0
    short_slope_days: int=10
    short_regime_exit: bool=True
    short_trail_atr: float=4.
    short_lock_profit: float=1.
    short_vol_target: float=0.
    short_signal_mode: int=0
    short_shock_atr: float=0.
    short_shock_bars: int=6
    only_shorts: bool=False
    regime_exit: bool=False
    hurst_min: float=0.
    long_adx_min: float=20
    short_adx_min: float=0.
    short_hurst_min: float=0.
    channel_close: bool=False
    pullback_rsi: float=0.
    dd_soft: float=.25
    dd_hard: float=.3
    cooldown_bars: int=2
    max_hold_bars: int=5760
    cost: float=.0015
    carry: float=.05
    delay: int=0
    long_short: bool=True

class Features:
    def __init__(self,d):
        self.d=d;self.cache={};self.day=resample(d,'1D')
    def get(self,p):
        per=int(pd.Timedelta('1D')/pd.Timedelta(p.rule))
        key=(p.rule,p.offset_hours,p.vol_fast_days,p.vol_slow_days)
        if key not in self.cache:
            b=resample(self.d,p.rule,pd.Timedelta(hours=p.offset_hours));per=int(pd.Timedelta('1D')/pd.Timedelta(p.rule))
            lr=np.log(b.close).diff()
            v=lr.rolling(p.vol_slow_days*per).std()*np.sqrt(365*per)
            b['vol']=pd.concat([lr.rolling(p.vol_fast_days*per).std()*np.sqrt(365*per),v],axis=1).max(axis=1)
            b['rank']=v.rolling(90*per,min_periods=30*per).rank(pct=True)
            logp=np.log(b.close)
            v1=logp.diff().rolling(90*per,min_periods=30*per).var()
            v16=logp.diff(16).rolling(90*per,min_periods=30*per).var()
            b['hurst']=(np.log(v16/v1)/(2*np.log(16))).clip(0,1)
            up=b.high.diff();dn=-b.low.diff();plus=up.where((up>dn)&(up>0),0.).ewm(alpha=1/14,adjust=False).mean();minus=dn.where((dn>up)&(dn>0),0.).ewm(alpha=1/14,adjust=False).mean()
            b['adx']=(100*(plus-minus).abs()/(plus+minus)).ewm(alpha=1/14,adjust=False,min_periods=28).mean()
            self.cache[key]=b
        b=self.cache[key];ma=self.day.close.rolling(p.regime_days).mean()
        bull=(self.day.close>ma).reindex(b.index,method='ffill',fill_value=False)
        sma=self.day.close.rolling(p.short_regime_days or p.regime_days).mean()
        bear=((self.day.close<sma)&(sma<sma.shift(p.short_slope_days))).reindex(b.index,method='ffill',fill_value=False)
        if p.short_shock_atr>0:
            fast=b.close.ewm(span=12,adjust=False).mean()
            shock=(b.close-b.close.shift(p.short_shock_bars)<-p.short_shock_atr*b.atr.shift(p.short_shock_bars))&(b.close<fast)&(fast<fast.shift(3))
            bear= bear|shock
        en=((b.close>(b.close if p.channel_close else b.high).shift().rolling(p.breakout).max())|((b.rsi<p.pullback_rsi)&(p.pullback_rsi>0)))&bull&(b['rank']<=p.vol_rank_max)&(b.hurst>=p.hurst_min)&(b.adx>=p.long_adx_min)
        if not p.weekends:en&=b.index.dayofweek<5
        sn=(b.close<b.low.shift().rolling(p.short_breakout).min())&bear&(b['rank']<=p.short_vol_rank)&(b.hurst>=p.short_hurst_min)&(b.adx>=p.short_adx_min)
        if p.short_signal_mode==1:sn=(b.close<b.close.ewm(span=p.short_breakout,adjust=False).mean())&(b.close<b.close.shift(3))&bear&(b['rank']<=p.short_vol_rank)&(b.hurst>=p.short_hurst_min)&(b.adx>=p.short_adx_min)
        if p.short_mode==2:sn&=b.close<b.close.ewm(span=20,adjust=False).mean()
        if p.short_mode==0 or not p.long_short:sn&=False
        lx=b.close<b.low.shift().rolling(p.exit_bars).min()
        sx=(b.close>b.high.shift().rolling(p.short_exit).max())|(~bear & p.short_regime_exit)
        if p.regime_exit:lx|=~bull
        if p.trend_mode:
            score=pd.concat([np.sign(b.close/b.close.shift(int(p.momentum_days*k*per))-1) for k in (.5,1.,2.)],axis=1).mean(axis=1)
            en=(score>=p.entry_threshold)&bull&(b['rank']<=p.vol_rank_max)&(b.hurst>=p.hurst_min)&(b.adx>=p.long_adx_min)
            sn=(score<=-p.entry_threshold)&bear&(b.hurst>=p.short_hurst_min)&(b.adx>=p.short_adx_min)&(p.long_short)&(p.short_mode!=0)
            lx=score<=p.exit_threshold; sx=(score>=-p.exit_threshold)|~bear
            if p.trend_mode==2: en=bull;sn=bear&(p.long_short);lx=~bull;sx=~bear
            if not p.weekends: en &= b.index.dayofweek<5
        if p.only_shorts:en &= False
        frame=pd.DataFrame(dict(en=en,sn=sn,lx=lx,sx=sx,valid=b.complete,atr=b.atr,size=pd.Series(np.where(en,p.vol_target,p.short_vol_target or p.vol_target),index=b.index).div(b.vol).clip(upper=1),hurst=b.hurst,vol=b.vol,vol_rank=b['rank'],adx=b.adx,bull=bull,bear=bear,short_break=(b.close<b.low.shift().rolling(p.short_breakout).min()),short_below_ma=(self.day.close<sma).reindex(b.index,method='ffill',fill_value=False),short_slope=(sma<sma.shift(p.short_slope_days)).reindex(b.index,method='ffill',fill_value=False)))
        aligned=frame.reindex(self.d.index)
        if p.delay:aligned=aligned.shift(p.delay)
        self.update_clock=aligned.valid.eq(True).to_numpy(bool)
        pending=None
        arr=aligned.to_numpy(copy=True); vi=aligned.columns.get_loc('valid')
        liquid=self.d.tradable.to_numpy()
        for i in range(len(arr)):
            if arr[i,vi] is True or arr[i,vi]==True: pending=arr[i].copy()
            if pending is not None:
                if liquid[i]: arr[i]=pending;pending=None
                else: arr[i,vi]=False
        aligned=pd.DataFrame(arr,index=aligned.index,columns=aligned.columns)
        flags=[aligned[k].eq(True).to_numpy(bool) for k in ['en','sn','lx','sx','valid']]
        vals=[pd.to_numeric(aligned[k]).fillna(0).to_numpy(float) for k in ['atr','size']]
        return (*flags,*vals),frame

@njit(cache=True)
def simulate(o,h,l,c,liquid,en,sn,lx,sx,valid,atr,size,start,end,initial,cost,carry,stop_atr,lock_trigger,lock_profit,trail_atr,short_cap,short_stop,short_lock,ddsoft,ddhard,cooldown,maxhold,update_clock=None,short_trail_atr=0.,short_lock_profit=-1.):
    n=len(c);eq=np.full(n,np.nan);exp=np.zeros(n);cp=np.full(n,np.nan);qp=np.zeros(n);dd=np.zeros(n);cy=np.zeros(n)
    stop_path=np.full(n,np.nan)
    orders=np.zeros((2*n+4,8));trades=np.zeros((n+2,7));oc=tc=0
    cash=initial;q=0.;stop=entry=ea=0.;peak=initial;born=start;until=start;ep_base=initial;fees=ecarry=totalcarry=0.;locked=False
    maxgross=0.;gaps=intrabr=0;recovering=False;risk_pending=False
    last=end-1
    for i in range(start,end):
        if i>start and q<0:
            charge=abs(q)*c[i-1]*carry/35040.;cash-=charge;totalcarry+=charge;ecarry+=charge
        pre=cash+q*o[i];gross=abs(q)*o[i]/max(pre,1e-12);maxgross=max(maxgross,gross)
        if gross>1+1e-9:gaps+=1
        localdd=1-pre/max(peak,1e-12)
        if localdd<=ddsoft:recovering=False
        risktrip=localdd>ddhard and not recovering
        if risktrip:recovering=True;risk_pending=q!=0;until=max(until,i+288)
        reason=0;close=False
        if q!=0 and liquid[i]:
            if (q>0 and o[i]<=stop) or (q<0 and o[i]>=stop):close=True;reason=1
            elif valid[i] and ((q>0 and lx[i]) or (q<0 and sx[i])):close=True;reason=0
            elif valid[i] and ((q>0 and sn[i]) or (q<0 and en[i])):close=True;reason=8
            elif i-born>=maxhold:close=True;reason=4
            elif risk_pending:close=True;reason=5
            elif gross>=.995 and q<0:close=True;reason=6
        if i==last and q!=0 and liquid[i]:close=True;reason=7
        if close:
            risk_pending=False
            side=np.sign(q);delta=-q;fee=abs(q)*o[i]*cost;cash-=delta*o[i]+fee;fees+=fee
            orders[oc]=np.array([i,delta,o[i],fee,reason,q,0.,cash]);oc+=1
            trades[tc]=np.array([born,i,side,cash-ep_base,fees,ecarry,(i-born)/96.]);tc+=1;q=0.
            if reason in (1,4,5,6):until=max(until,i+cooldown)
        if q==0 and i<last and i>=until and valid[i] and liquid[i] and atr[i]>0:
            side=1. if en[i] else (-1. if sn[i] else 0.)
            if side!=0:
                fraction=min(1.,size[i]) if side>0 else min(short_cap,size[i])
                localdd=1-cash/max(peak,1e-12)
                if localdd>ddsoft:fraction*=.5
                if localdd>ddhard:fraction*=.5
                ep_base=cash;fees=ecarry=0.;born=i;entry=o[i];ea=atr[i];locked=False
                q=side*fraction*cash/(o[i]*(1+fraction*cost));fee=abs(q)*o[i]*cost;cash-=q*o[i]+fee;fees+=fee
                stop=entry-side*(stop_atr if side>0 else short_stop)*ea
                orders[oc]=np.array([i,q,o[i],fee,9,0.,q,cash]);oc+=1
        if q!=0:
            side=np.sign(q);price=stop;reason=2
            if q<0:
                capprice=cash/(abs(q)*(2+cost))*.995
                if capprice<price:price=capprice;reason=6
            trigger=liquid[i] and ((q>0 and l[i]<=price) or (q<0 and h[i]>=price))
            if trigger:price=min(o[i],price) if q>0 else max(o[i],price)
            adverse=h[i] if q>0 or not trigger else price
            wg=abs(q)*adverse/max(cash+q*adverse,1e-12);maxgross=max(maxgross,wg)
            if wg>1+1e-9:intrabr+=1
            if trigger:
                delta=-q;fee=abs(q)*price*cost;cash-=delta*price+fee;fees+=fee
                orders[oc]=np.array([i,delta,price,fee,reason,q,0.,cash]);oc+=1
                trades[tc]=np.array([born,i,side,cash-ep_base,fees,ecarry,(i-born)/96.]);tc+=1
                q=0.;until=i+cooldown
            elif i+1<end and (valid[i+1] if update_clock is None else update_clock[i+1]):
                profit=side*(c[i]-entry);threshold=lock_trigger if side>0 else short_lock
                if not locked and profit>=threshold*ea:
                    locked=True;floor=short_lock_profit if side<0 and short_lock_profit>=0 else lock_profit
                    level=entry+side*max(floor*ea,entry*2.2*cost)
                    stop=max(stop,level) if side>0 else min(stop,level)
                trailing=short_trail_atr if side<0 else trail_atr
                if trailing>0 and locked:
                    level=c[i]-side*trailing*ea;stop=max(stop,level) if side>0 else min(stop,level)
        stop_path[i]=stop if q!=0 else np.nan
        equity=cash+q*c[i];peak=max(peak,equity);eq[i]=equity;exp[i]=abs(q)*c[i]/max(equity,1e-12);cp[i]=cash;qp[i]=q;dd[i]=1-equity/peak;cy[i]=totalcarry
    return eq,exp,cp,qp,dd,cy,orders[:oc],trades[:tc],maxgross,gaps,intrabr,stop_path

def run(d,p,features=None,start=None,end=None,initial=INITIAL):
    if not (0<=p.short_cap<1 and p.cost>=0 and p.carry>=0 and p.vol_target>0):raise ValueError('Invalid configuration')
    f=features or Features(d);args,diagnostics=f.get(p)
    a=0 if start is None else int(d.index.searchsorted(pd.Timestamp(start,tz='UTC')))
    z=len(d) if end is None else int(d.index.searchsorted(pd.Timestamp(end,tz='UTC')))
    if z-a<2:raise ValueError('Empty window')
    x=simulate(*(d[k].to_numpy() for k in ['open','high','low','close','tradable']),*args,a,z,initial,p.cost,p.carry,p.stop_atr,p.lock_trigger,p.lock_profit,p.trail_atr,p.short_cap,p.short_stop,p.short_lock,p.dd_soft,p.dd_hard,p.cooldown_bars,p.max_hold_bars,f.update_clock,p.short_trail_atr,p.short_lock_profit)
    path=pd.DataFrame(dict(equity=x[0],exposure=x[1],cash=x[2],units=x[3],local_drawdown=x[4],carry=x[5]),index=d.index).iloc[a:z]
    path['standing_stop_next_bar']=x[11][a:z]
    path['signed_exposure']=path.units*d.close.reindex(path.index)/path.equity
    fills=pd.DataFrame(x[6],columns=['bar','delta_units','price','cost','reason','closed_units','result_units','cash_after']);fills['timestamp']=d.index[fills.bar.astype(int)]
    tr=pd.DataFrame(x[7],columns=['entry_bar','exit_bar','direction','pnl','cost','carry','days']);tr['entry_time']=d.index[tr.entry_bar.astype(int)];tr['exit_time']=d.index[tr.exit_bar.astype(int)]
    if abs(path.units.iloc[-1])<1e-8 and abs(path.equity.iloc[-1]-initial-tr.pnl.sum())>1e-5:raise AssertionError('Episode ledger mismatch')
    path['restricted_collateral']=2*abs(path.units.clip(upper=0))*d.close.reindex(path.index)
    path['free_cash']=path.cash-path.restricted_collateral
    audit=dict(max_observed_gross=float(x[8]),gap_cap_breaches=int(x[9]),intrabar_cap_breaches=int(x[10]),all_fills_tradable=bool(d.tradable.reindex(fills.timestamp).all()),terminal_flat=bool(abs(path.units.iloc[-1])<1e-8))
    return path,fills,tr,audit

def load_data(file):
    raw = pd.read_csv(file)
    tc = next((x for x in ('timestamp','open_time','time','datetime','date') if x in raw),None)
    if tc is None: raise ValueError('Missing timestamp column')
    idx = pd.to_datetime(raw[tc].astype(str),format='ISO8601',utc=True)
    if idx.duplicated().any(): raise ValueError('Duplicate timestamps')
    d = raw[['open','high','low','close','volume']].apply(pd.to_numeric,errors='raise')
    d.index = idx
    if not np.isfinite(d.to_numpy()).all(): raise ValueError('Nonfinite OHLCV')
    if (d[['open','high','low','close']]<=0).any().any() or (d.volume<0).any(): raise ValueError('Invalid prices/volume')
    if ((d.high<d[['open','close','low']].max(axis=1))|(d.low>d[['open','close','high']].min(axis=1))).any(): raise ValueError('OHLC envelope violation')
    filled = pd.Series(False,index=idx)
    if 'is_filled' in raw:
        flags = raw.is_filled.astype(str).str.lower()
        if not flags.isin(['true','false','0','1']).all(): raise ValueError('Invalid is_filled flag')
        filled = pd.Series(flags.isin(['true','1']).to_numpy(),index=idx)
    d['tradable'] = (d.volume>0)&~filled
    d = d.sort_index()
    offgrid = d.index != d.index.floor('15min')
    audit = dict(raw_bars=len(d),off_grid_excluded=int(offgrid.sum()),zero_volume=int((d.volume==0).sum()),filled_flag=int(filled.sum()))
    d = d.loc[~offgrid]
    if d.index.to_series().diff().dropna().median()!=pd.Timedelta('15min'): raise ValueError('Expected 15-minute data; refusing coarse-data upsampling')
    if not len(d): raise ValueError('No 15-minute bars')
    grid = pd.date_range(d.index[0],d.index[-1],freq='15min')
    d = d.reindex(grid)
    missing = d.close.isna()
    previous = d.close.ffill()
    for c in ('open','high','low','close'): d[c] = d[c].where(~missing,previous)
    d['volume'] = d.volume.fillna(0)
    d['tradable'] = d.tradable.eq(True)
    d.index.name = 'timestamp'
    big=[]
    if missing.any():
        g=(missing!=missing.shift()).cumsum()
        for _,x in missing.groupby(g):
            if x.iloc[0] and len(x)>=7*96:big.append([str(x.index[0]),str(x.index[-1])])
    audit.update(big_gaps=big,grid_bars=len(d),missing_synthetic=int(missing.sum()),untradable=int((~d.tradable).sum()),start=str(d.index[0]),end=str(d.index[-1]))
    d.attrs['audit'] = audit
    return d

def resample(d,rule,offset=None):
    b = d.resample(rule,origin='start_day',offset=offset).agg({'open':'first','high':'max','low':'min','close':'last','volume':'sum','tradable':'sum'})
    b.index += pd.Timedelta(rule)
    expected = int(pd.Timedelta(rule)/pd.Timedelta('15min'))
    b['complete'] = b.tradable==expected
    prev=b.close.shift()
    tr=pd.concat([b.high-b.low,(b.high-prev).abs(),(b.low-prev).abs()],axis=1).max(axis=1)
    b['atr']=tr.ewm(alpha=1/14,adjust=False,min_periods=14).mean()
    r=b.close.diff();up=r.clip(lower=0).ewm(alpha=1/3,adjust=False,min_periods=3).mean();dn=(-r.clip(upper=0)).ewm(alpha=1/3,adjust=False,min_periods=3).mean()
    b['rsi']=(100*up/(up+dn)).fillna(50)
    return b

def benchmark(d,cost=.0015,initial=INITIAL):
    ix=np.flatnonzero(d.tradable.to_numpy())
    if not len(ix) or not d.tradable.iloc[-1]: raise ValueError('Benchmark terminal open unavailable; no retrospective terminal fill')
    a,z=ix[0],len(d)-1
    qty=initial/(d.open.iloc[a]*(1+cost));out=pd.Series(initial,index=d.index,dtype=float)
    out.iloc[a:z+1]=qty*d.close.iloc[a:z+1]
    out.iloc[z:]=qty*d.open.iloc[z]*(1-cost)
    return out

def quarters(eq,bh,initial=INITIAL):
    rows=[];prev=prevbh=initial
    for period,s in eq.groupby(eq.index.tz_localize(None).to_period('Q')):
        b=bh.reindex(s.index);complete=s.index[0]<=period.start_time.tz_localize('UTC') and s.index[-1]>=period.end_time.floor('15min').tz_localize('UTC')
        sr=s.iloc[-1]/prev-1;br=b.iloc[-1]/prevbh-1
        rows.append(dict(quarter=str(period),strategy_return_pct=100*sr,buy_hold_return_pct=100*br,excess_pct=100*(sr-br),beat=bool(sr>br),complete=bool(complete)))
        prev=s.iloc[-1];prevbh=b.iloc[-1]
    return pd.DataFrame(rows)

def metrics(path,fills,tr,bh,audit,initial=INITIAL):
    eq=path.equity;days=(len(eq)*15/1440);daily=eq.resample('1D').last();r=daily.pct_change();r.iloc[0]=daily.iloc[0]/initial-1;r=r.fillna(0)
    downside=np.sqrt(np.mean(np.minimum(r,0)**2));sh=np.sqrt(365)*r.mean()/r.std(ddof=1) if r.std()>0 else 0.
    dd=eq/pd.Series(np.maximum.accumulate(np.r_[initial,eq.to_numpy()])[1:],index=eq.index)-1
    under=dd<-1e-10;longest=cur=0
    for u in under:cur=cur+1 if u else 0;longest=max(longest,cur)
    q=quarters(eq,bh,initial);qc=q[q.complete];wins=tr.loc[tr.pnl>0,'pnl'];loss=tr.loc[tr.pnl<=0,'pnl']
    m=dict(total_return_pct=float(100*(eq.iloc[-1]/initial-1)),short_trades=int((tr.direction<0).sum()),same_open_reversals=int(sum(len(g)==2 and g.iloc[0].closed_units*g.iloc[1].result_units<0 for _,g in fills.groupby('bar'))),cagr_pct=100*((eq.iloc[-1]/initial)**(365.25/days)-1),net_profit=float(eq.iloc[-1]-initial),final_equity=float(eq.iloc[-1]),sharpe=float(sh),sortino=float(np.sqrt(365)*r.mean()/downside) if downside>0 else 0.,max_drawdown_pct=float(-100*dd.min()),mean_exposure_pct=float(100*path.exposure.mean()),time_in_market_pct=float(100*(path.exposure>1e-6).mean()),longest_underwater_days=longest/96.,cost_paid=float(fills.cost.sum()),carry_paid=float(path.carry.iloc[-1]),executed_orders=len(fills),total_closed_trades=len(tr),win_rate_pct=float(100*(tr.pnl>0).mean()) if len(tr) else 0.,gross_profit=float(wins.sum()),gross_loss=float(-loss.sum()),average_winning_trade=float(wins.mean()) if len(wins) else 0.,average_losing_trade=float(loss.mean()) if len(loss) else 0.,largest_winning_trade=float(wins.max()) if len(wins) else 0.,largest_losing_trade=float(loss.min()) if len(loss) else 0.,average_holding_days=float(tr.days.mean()) if len(tr) else 0.,maximum_holding_days=float(tr.days.max()) if len(tr) else 0.,buy_hold_return_pct=float(100*(bh.iloc[-1]/initial-1)),quarter_beats=int(qc.beat.sum()),complete_quarters=len(qc),quarter_beat_pct=float(100*qc.beat.mean()) if len(qc) else 0.,daily_observations=len(r),**audit)
    gp=tr.pnl+tr.cost+tr.carry
    m.update(gross_profit_before_cost=float(gp.clip(lower=0).sum()),gross_loss_before_cost=float(-gp.clip(upper=0).sum()),pnl_before_cost_total=float(gp.sum()),annual_realized_vol_pct=float(r.std(ddof=1)*np.sqrt(365)*100),one_way_turnover_equity_pa=float((fills.delta_units.abs()*fills.price).sum()/eq.mean()/(days/365.25)),cost_pct_average_equity_pa=float(100*fills.cost.sum()/eq.mean()/(days/365.25)),short_episode_net_pnl=float(tr.loc[tr.direction<0,'pnl'].sum()),long_episode_net_pnl=float(tr.loc[tr.direction>0,'pnl'].sum()))
    return m,q,r

def export(dest,d,p,path,fills,tr,audit,label,initial=INITIAL):
    dest=Path(dest);dest.mkdir(parents=True,exist_ok=True);bh=benchmark(d.reindex(path.index),p.cost,initial)
    m,q,r=metrics(path,fills,tr,bh,audit,initial);m['result_type']=label
    path.assign(buy_hold=bh).to_csv(dest/'equity.csv');fills.to_csv(dest/'fills.csv',index=False);tr.to_csv(dest/'trades.csv',index=False);q.to_csv(dest/'quarters.csv',index=False)
    (dest/'metrics.json').write_text(json.dumps(m,indent=2));return m

FROZEN_PARAMETERS=asdict(Config())

WINDOWS = [
    ("2017_2020", None, "2021-01-01"),
    ("2021_2025", "2021-01-01", "2026-01-01"),
    ("2026_to_date", "2026-01-01", None),
    ("full", None, None),
]


def bh_stats(bh):
    daily = bh.resample("1D").last()
    r = daily.pct_change().fillna(0)
    years = len(bh) * 15 / 1440 / 365.25
    return {
        "bh_cagr_pct": float(((bh.iloc[-1] / bh.iloc[0]) ** (1 / years) - 1) * 100),
        "bh_sharpe": float(r.mean() / r.std() * np.sqrt(365)) if r.std() > 0 else 0.0,
        "bh_max_drawdown_pct": float(-(bh / bh.cummax() - 1).min() * 100),
    }


def plot_window(dest, d, path, tr, label):
    bh = INITIAL * d.close.reindex(path.index) / d.open.reindex(path.index).iloc[0]
    eq = path.equity
    fig, ax = plt.subplots(4, 1, figsize=(14, 15), sharex=True, gridspec_kw={"height_ratios": [3, 1.2, 1.2, 2.2]})
    ax[0].plot(eq.index, eq.values, lw=1.2, label="Strategy")
    ax[0].plot(bh.index, bh.values, lw=0.9, alpha=0.7, label="Buy & hold")
    ax[0].set_yscale("log")
    ax[0].set_title(f"ETHUSDT {label}: equity vs buy & hold (log, 0.15% per side, no leverage)")
    ax[0].legend()
    ax[0].grid(alpha=0.3)
    ax[1].fill_between(eq.index, (eq / eq.cummax() - 1) * 100, 0, color="tab:red", alpha=0.5, label="Strategy")
    ax[1].plot(bh.index, (bh / bh.cummax() - 1) * 100, color="gray", lw=0.7, label="Buy & hold")
    ax[1].set_ylabel("Drawdown %")
    ax[1].legend()
    ax[1].grid(alpha=0.3)
    ax[2].fill_between(path.index, path.signed_exposure.values, 0, color="tab:purple", alpha=0.6, step="post")
    ax[2].set_ylabel("Signed exposure")
    ax[2].set_ylim(-1.05, 1.05)
    ax[2].grid(alpha=0.3)
    px = d.close.reindex(path.index)
    ax[3].plot(px.index, px.values, color="black", lw=0.5)
    ax[3].set_yscale("log")
    if len(tr):
        for side, col, mk, nm in ((1, "tab:green", "^", "Long entry"), (-1, "tab:red", "v", "Short entry")):
            t = tr[tr.direction == side]
            ax[3].scatter(t.entry_time, d.open.reindex(t.entry_time).values, s=14, color=col, marker=mk, label=nm)
    ax[3].set_title("Trade history")
    ax[3].legend()
    ax[3].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(Path(dest) / "backtest.png", dpi=105)
    plt.close(fig)


def run_window(d, f, p, name, a, b, out):
    path, fills, tr, audit = run(d, p, features=f, start=a, end=b)
    if audit["gap_cap_breaches"] or audit["intrabar_cap_breaches"] or (path.equity <= 0).any() or not audit["terminal_flat"]:
        (out / f"rejection_audit_{name}.json").write_text(json.dumps(audit, indent=2))
        raise SystemExit(f"Replay rejected for {name}: observed >1x gross exposure or nonpositive equity.")
    dest = out / name
    m = export(dest, d, p, path, fills, tr, audit, name)
    bh = benchmark(d.reindex(path.index), p.cost)
    m.update(bh_stats(bh))
    (dest / "metrics.json").write_text(json.dumps(m, indent=2))
    plot_window(dest, d, path, tr, name)
    return m


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--out", default="eth_results")
    ap.add_argument("--start")
    ap.add_argument("--end")
    ap.add_argument("--long-only", action="store_true")
    ap.add_argument("--cost", type=float)
    ap.add_argument("--carry", type=float)
    ap.add_argument("--delay", type=int)
    args = ap.parse_args()
    p = Config(**FROZEN_PARAMETERS)
    for k in ["cost", "carry", "delay"]:
        if getattr(args, k) is not None:
            p = replace(p, **{k: getattr(args, k)})
    if args.long_only:
        p = replace(p, long_short=False)
    if p.cost < .0015 or p.delay < 0 or p.carry < 0:
        ap.error("Cost must be >=0.15% per side; delay and carry nonnegative")
    np.random.seed(26)
    d = load_data(args.data)
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    audit = d.attrs["audit"]
    (out / "data_audit.json").write_text(json.dumps(audit, indent=2))
    (out / "used_config.json").write_text(json.dumps(asdict(p), indent=2))
    f = Features(d)
    wins = [("custom", args.start, args.end)] if (args.start or args.end) else list(WINDOWS)
    expanded = []
    for name, a, b in wins:
        expanded.append((name, a, b))
        lo = d.index[0] if a is None else pd.Timestamp(a, tz="UTC")
        hi = d.index[-1] + pd.Timedelta("15min") if b is None else pd.Timestamp(b, tz="UTC")
        for gs, ge in audit["big_gaps"]:
            gs = pd.Timestamp(gs)
            if lo < gs < hi:
                expanded.append((f"{name}_before_data_gap", a, gs.strftime("%Y-%m-%d %H:%M")))
    rows = {}
    for name, a, b in expanded:
        rows[name] = run_window(d, f, p, name, a, b, out)
    keys = ["total_return_pct", "cagr_pct", "sharpe", "sortino", "max_drawdown_pct", "win_rate_pct", "total_closed_trades", "short_trades", "average_holding_days", "quarter_beats", "complete_quarters", "quarter_beat_pct", "buy_hold_return_pct", "bh_cagr_pct", "bh_sharpe", "bh_max_drawdown_pct", "cost_paid", "carry_paid", "max_observed_gross"]
    summary = pd.DataFrame({k: {x: rows[k][x] for x in keys} for k in rows}).T
    summary.to_csv(out / "summary.csv")
    (out / "summary.json").write_text(json.dumps({k: {x: rows[k][x] for x in keys} for k in rows}, indent=2))
    print(summary.round(2).T.to_string())
    if audit["big_gaps"]:
        print("Data gaps of 7 days or more (untradable, no fills):", audit["big_gaps"])


if __name__ == "__main__":
    main()

### Load modules

In [ ]:
def load_module(name, fname):
    p = DATA_DIR / fname
    if not p.exists():
        raise FileNotFoundError(f"{p.resolve()} not found - put the notebook next to your .py/.csv files or edit DATA_DIR")
    spec = importlib.util.spec_from_file_location(name, p)
    mod = importlib.util.module_from_spec(spec); sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod

btc_mod = load_module("btc_strat", "bitcoin_crash_strategy.py")
eth_mod = load_module("eth_strat", "eth_final_strategy.py")

%matplotlib inline

SPEC = {
    "BTC": dict(mod=btc_mod, csv="BTCUSDT.csv", cfg=btc_mod.Config(**btc_mod.FROZEN_PARAMETERS)),
    "ETH": dict(mod=eth_mod, csv="ETHUSDT.csv", cfg=eth_mod.Config(**eth_mod.FROZEN_PARAMETERS)),
}
SPEC = {a: SPEC[a] for a in ASSETS}
print("Strategy modules loaded for:", list(SPEC))

## 3 · Historical baseline

In [ ]:
def equity_metrics(eq, initial=INITIAL):
    eq = eq.dropna()
    daily = eq.resample("1D").last().dropna()
    r = daily.pct_change().dropna()
    days = max((eq.index[-1] - eq.index[0]).total_seconds() / 86400, 1)
    peak = np.maximum.accumulate(np.r_[initial, eq.to_numpy()])[1:]
    final = eq.iloc[-1]
    return dict(total_return_pct=100 * (final / initial - 1),
                cagr_pct=100 * ((max(final, 1e-9) / initial) ** (365.25 / days) - 1),
                sharpe=float(np.sqrt(365) * r.mean() / r.std(ddof=1)) if r.std() > 0 else 0.0,
                max_dd_pct=float(100 * (1 - eq.to_numpy() / peak).max()))

def bh_curve(d, idx, initial=INITIAL):
    return initial * d.close.reindex(idx) / d.open.reindex(idx).iloc[0]

def run_strategy(a, d=None, cfg=None, features=None, start=None, end=None):
    s = SPEC[a]; d = s["d"] if d is None else d
    return s["mod"].run(d, s["cfg"] if cfg is None else cfg, features=features, start=start, end=end)

rows = {}
for a, s in SPEC.items():
    t0 = time.time()
    s["d"] = s["mod"].load_data(DATA_DIR / s["csv"])
    s["F"] = s["mod"].Features(s["d"])
    s["path"], s["fills"], s["tr"], s["audit"] = run_strategy(a, features=s["F"])
    s["bh"] = bh_curve(s["d"], s["path"].index)
    m, b = equity_metrics(s["path"].equity), equity_metrics(s["bh"])
    rows[f"{a} strategy"] = {**m, "trades": len(s["tr"]), "short_trades": int((s["tr"].direction < 0).sum()),
                             "win_rate_pct": 100 * (s["tr"].pnl > 0).mean()}
    rows[f"{a} buy&hold"] = b
    print(f"{a}: {len(s['d']):,} bars  {s['d'].index[0].date()} -> {s['d'].index[-1].date()}   ({time.time()-t0:.1f}s)   audit ok: "
          f"{s['audit']['gap_cap_breaches']==0 and s['audit']['intrabar_cap_breaches']==0}")
baseline = pd.DataFrame(rows).T
display(baseline.round(2))

fig, axes = plt.subplots(2, len(SPEC), figsize=(7 * len(SPEC), 7), sharex="col", squeeze=False,
                         gridspec_kw={"height_ratios": [3, 1]})
for j, (a, s) in enumerate(SPEC.items()):
    eq = s["path"].equity
    axes[0, j].plot(eq, color=C_STRAT, lw=1.1, label="Strategy"); axes[0, j].plot(s["bh"], color=C_BH, lw=.9, label="Buy & hold")
    axes[0, j].set_yscale("log"); axes[0, j].set_title(f"{a}: historical backtest (log scale)"); axes[0, j].legend()
    axes[1, j].fill_between(eq.index, -100 * (1 - eq / eq.cummax()), 0, color=C_BAD, alpha=.5, label="Strategy DD")
    axes[1, j].plot(-100 * (1 - s["bh"] / s["bh"].cummax()), color=C_BH, lw=.7, label="B&H DD")
    axes[1, j].set_ylabel("Drawdown %"); axes[1, j].legend(loc="lower left")
plt.tight_layout(); plt.show()

---
## Part A — Monte Carlo: trade resampling
Take the strategy's actual closed trades, convert each to a **return on the equity it was opened with**, and re-draw the sequence thousands of times:
* **Bootstrap** – sample trades with replacement (different mix of winners/losers)
* **Shuffle** – same trades, random order (isolates *sequence* risk → drawdown)
* **Block bootstrap** – bootstrap in blocks of `BLOCK` trades (keeps streaks / clustering)
* **Skip 15 %** – bootstrap, then randomly miss `SKIP_FRAC` of trades (missed fills / downtime)

In [ ]:
def trade_returns(tr, initial=INITIAL):
    pnl = tr.pnl.to_numpy()
    return pnl / (initial + np.r_[0.0, np.cumsum(pnl)[:-1]])

dep_rows = {}
for a, s in SPEC.items():
    s["r"] = trade_returns(s["tr"])
    chk = INITIAL * np.prod(1 + s["r"])
    assert abs(chk - s["path"].equity.iloc[-1]) < 1.0, "trade-return reconstruction does not match final equity"
    srt = np.sort(s["r"])[::-1]
    dep = {"all trades": 100 * (np.prod(1 + s["r"]) - 1)}
    for k in (1, 3, 5):
        dep[f"without best {k}"] = 100 * (np.prod(1 + srt[k:]) - 1)
    dep_rows[a] = dep
    print(f"{a}: {len(s['r'])} trades | mean {100*s['r'].mean():.2f}% | median {100*np.median(s['r']):.2f}% | "
          f"best {100*s['r'].max():.1f}% | worst {100*s['r'].min():.1f}% | win-rate {100*(s['r']>0).mean():.0f}%")
print("\nTotal return (%) after removing the best N trades  -> how fragile is the edge to a few outliers?")
display(pd.DataFrame(dep_rows).T.round(1))

In [ ]:
METHODS = ["Bootstrap", "Shuffle", "Block bootstrap", f"Skip {int(SKIP_FRAC*100)}%"]

def resample_trades(r, method, n_sims, rng):
    n = len(r)
    if method == "Shuffle":
        idx = np.argsort(rng.random((n_sims, n)), axis=1); return r[idx]
    if method == "Block bootstrap":
        nb = int(np.ceil(n / BLOCK)); st = rng.integers(0, n, (n_sims, nb))
        idx = ((st[:, :, None] + np.arange(BLOCK)) % n).reshape(n_sims, -1)[:, :n]; return r[idx]
    rr = r[rng.integers(0, n, (n_sims, n))]
    if method.startswith("Skip"):
        rr = np.where(rng.random(rr.shape) < SKIP_FRAC, 0.0, rr)
    return rr

def path_stats(rr):
    eq = INITIAL * np.cumprod(1 + rr, axis=1)
    eqf = np.concatenate([np.full((len(rr), 1), INITIAL), eq], axis=1)
    dd = 1 - eqf / np.maximum.accumulate(eqf, axis=1)
    return eqf, eq[:, -1] / INITIAL - 1, dd.max(axis=1)

rng = np.random.default_rng(SEED)
tables = {}
for a, s in SPEC.items():
    s["mc"] = {}
    for m in METHODS:
        rr = resample_trades(s["r"], m, N_TRADE_SIMS, rng)
        s["mc"][m] = path_stats(rr)
    _, orig_ret, orig_dd = path_stats(s["r"][None, :])
    rows = {"Actual (trade-close basis)": dict(median_ret=100*orig_ret[0], p5_ret=np.nan, p95_ret=np.nan, prob_loss=np.nan,
                                              median_dd=100*orig_dd[0], p95_dd=np.nan, p_dd_gt30=np.nan, p_dd_gt50=np.nan)}
    for m, (eqf, ret, dd) in s["mc"].items():
        rows[m] = dict(median_ret=100*np.median(ret), p5_ret=100*np.percentile(ret, 5), p95_ret=100*np.percentile(ret, 95),
                       prob_loss=100*(ret < 0).mean(), median_dd=100*np.median(dd), p95_dd=100*np.percentile(dd, 95),
                       p_dd_gt30=100*(dd > .30).mean(), p_dd_gt50=100*(dd > .50).mean())
    tables[a] = pd.DataFrame(rows).T
    s["mc_orig"] = (orig_ret[0], orig_dd[0])
    print(f"\n{a} — {N_TRADE_SIMS:,} simulated sequences per method  (all numbers in %)")
    display(tables[a].round(1))
pd.concat(tables, names=["asset"]).to_csv(OUT_DIR / "A_trade_resampling.csv")

In [ ]:
mcol = dict(zip(METHODS, ["#1f77b4", "#2ca02c", "#9467bd", "#ff7f0e"]))
for a, s in SPEC.items():
    fig, ax = plt.subplots(1, 3, figsize=(18, 4.8))
    eqf = s["mc"]["Bootstrap"][0]; x = np.arange(eqf.shape[1])
    for p in eqf[:120]: ax[0].plot(x, p, color="grey", alpha=.12, lw=.6)
    ax[0].fill_between(x, *np.percentile(eqf, [5, 95], axis=0), color=C_STRAT, alpha=.18, label="5–95 %")
    ax[0].fill_between(x, *np.percentile(eqf, [25, 75], axis=0), color=C_STRAT, alpha=.30, label="25–75 %")
    ax[0].plot(x, np.median(eqf, axis=0), color=C_STRAT, lw=2, label="Median")
    ax[0].plot(INITIAL * np.r_[1, np.cumprod(1 + s["r"])], color=C_BASE, lw=2, label="Actual order")
    ax[0].set_yscale("log"); ax[0].set_xlabel("Trade #"); ax[0].set_title(f"{a}: bootstrap equity fan"); ax[0].legend()

    wealth = {m: v for m, v in s["mc"].items() if m != "Shuffle"}
    allret = np.concatenate([1 + v[1] for v in wealth.values()])
    bins = np.geomspace(np.percentile(allret, .5), np.percentile(allret, 99.5), 55)
    for m, (_, ret, dd) in wealth.items():
        ax[1].hist(1 + ret, bins=bins, histtype="step", lw=1.6, color=mcol[m], label=m, density=True)
    ax[1].axvline(1 + s["mc_orig"][0], color=C_BASE, lw=2, label="Actual"); ax[1].axvline(1, color="k", lw=.8, ls="--")
    ax[1].set_xscale("log"); ax[1].set_xlabel("Final equity multiple (x)"); ax[1].set_title("Terminal wealth (Shuffle omitted: identical by construction)"); ax[1].legend(fontsize=8)

    top = max(np.percentile(v[2], 99.5) for v in s["mc"].values()) * 100
    bins = np.linspace(0, top, 45)
    for m, (_, ret, dd) in s["mc"].items():
        ax[2].hist(100 * dd, bins=bins, histtype="step", lw=1.6, color=mcol[m], label=m, density=True)
    ax[2].axvline(100 * s["mc_orig"][1], color=C_BASE, lw=2, label="Actual")
    ax[2].set_xlabel("Max drawdown % (trade-close basis)"); ax[2].set_title("Drawdown distribution"); ax[2].legend(fontsize=8)
    plt.tight_layout(); plt.show()

**Reading Part A**
* *Shuffle* has the **same** terminal wealth as the actual run by construction — only the drawdown changes. If the actual drawdown sits at the low end of the shuffled distribution, the historical path was luckier than average.
* *Bootstrap* / *Block* / *Skip* show how much the **final return** depends on the specific trades. `prob_loss` and `p5_ret` are the numbers to look at.
* Drawdown is measured on trade-close equity, so it is a little smaller than the intra-trade drawdown of the 15-min backtest.

---
## Part B — Monte Carlo: parameter perturbation
Every run multiplies each tunable parameter by a random factor in **[1 − `PERTURB_PCT`, 1 + `PERTURB_PCT`]** (integers are rounded, disabled features stay disabled, constraints such as `short_cap < 1` and `dd_soft < dd_hard` are enforced) and re-runs the full 2021–2025 backtest. If the frozen parameters sit on a sharp peak, most perturbed runs will be far worse than the baseline.

In [ ]:
INT_PARAMS   = ["regime_days", "breakout", "exit_bars", "short_breakout", "short_exit",
                "short_regime_days", "short_slope_days", "cooldown_bars"]
FLOAT_PARAMS = ["stop_atr", "lock_trigger", "lock_profit", "trail_atr", "vol_rank_max", "vol_target",
                "short_cap", "short_stop", "short_lock", "short_trail_atr", "short_lock_profit", "short_vol_target",
                "short_hurst_min", "long_adx_min", "short_hysteresis", "dd_soft", "dd_hard"]

def perturb(cfg, rng, pct):
    ch = {}
    for k in INT_PARAMS + FLOAT_PARAMS:
        v = getattr(cfg, k, None)
        if v is None or v <= 0:
            continue
        nv = v * (1 + rng.uniform(-pct, pct))
        ch[k] = max(1, int(round(nv))) if k in INT_PARAMS else float(nv)
    if "vol_rank_max" in ch: ch["vol_rank_max"] = min(ch["vol_rank_max"], 1.0)
    if "short_cap" in ch:    ch["short_cap"] = min(ch["short_cap"], 0.99)
    if "dd_soft" in ch:      ch["dd_soft"] = min(ch["dd_soft"], 0.97)
    if "dd_hard" in ch:      ch["dd_hard"] = min(max(ch["dd_hard"], ch.get("dd_soft", cfg.dd_soft) + 0.01), 0.99)
    return replace(cfg, **ch), {k: ch[k] / getattr(cfg, k) for k in ch}

print("Parameters that will be perturbed (non-zero in the frozen config):")
for a, s in SPEC.items():
    print(f"  {a}:", [k for k in INT_PARAMS + FLOAT_PARAMS if (getattr(s['cfg'], k, 0) or 0) > 0])

In [ ]:
rng = np.random.default_rng(SEED + 1)
PARAM = {}
for a, s in SPEC.items():
    rows, curves, t0 = [], {}, time.time()
    base_m = equity_metrics(s["path"].equity)
    rows.append(dict(run=0, kind="baseline", **base_m, trades=len(s["tr"])))
    curves[0] = s["path"].equity.resample("1D").last()
    for i in range(1, N_PARAM_RUNS + 1):
        cfg, mult = perturb(s["cfg"], rng, PERTURB_PCT)
        try:
            path, fills, tr, audit = s["mod"].run(s["d"], cfg, features=s["F"])
            if (path.equity <= 0).any(): raise ValueError("equity <= 0")
            rows.append(dict(run=i, kind="perturbed", **equity_metrics(path.equity), trades=len(tr),
                             **{f"x_{k}": v for k, v in mult.items()}))
            curves[i] = path.equity.resample("1D").last()
        except Exception as e:
            rows.append(dict(run=i, kind="failed", error=str(e)[:80]))
        if i % 25 == 0: print(f"  {a}: {i}/{N_PARAM_RUNS}  ({time.time()-t0:.0f}s)")
    df = pd.DataFrame(rows)
    PARAM[a] = dict(df=df, curves=curves, base=base_m)
    ok = df[df.kind == "perturbed"]
    summ = pd.Series({
        "runs ok": len(ok), "runs failed": int((df.kind == "failed").sum()),
        "baseline return %": base_m["total_return_pct"], "median return %": ok.total_return_pct.median(),
        "5th pct return %": ok.total_return_pct.quantile(.05), "worst return %": ok.total_return_pct.min(),
        "% runs profitable": 100 * (ok.total_return_pct > 0).mean(),
        "% runs beating B&H": 100 * (ok.total_return_pct > equity_metrics(s["bh"])["total_return_pct"]).mean(),
        "baseline Sharpe": base_m["sharpe"], "median Sharpe": ok.sharpe.median(), "5th pct Sharpe": ok.sharpe.quantile(.05),
        "% runs Sharpe >= 70% of baseline": 100 * (ok.sharpe >= .7 * base_m["sharpe"]).mean(),
        "baseline max DD %": base_m["max_dd_pct"], "median max DD %": ok.max_dd_pct.median(), "95th pct max DD %": ok.max_dd_pct.quantile(.95)})
    PARAM[a]["summary"] = summ
    print(f"\n=== {a}: parameter perturbation (+/-{int(PERTURB_PCT*100)} %, {len(ok)} runs) ===")
    display(summ.round(2).to_frame("value"))
    df.to_csv(OUT_DIR / f"B_param_perturbation_{a}.csv", index=False)

In [ ]:
for a, s in SPEC.items():
    P = PARAM[a]; ok = P["df"][P["df"].kind == "perturbed"]; base = P["base"]
    bh_ret = equity_metrics(s["bh"])["total_return_pct"]
    fig, ax = plt.subplots(2, 3, figsize=(18, 9))
    for i, c in P["curves"].items():
        if i: ax[0, 0].plot(c.index, c.values, color=C_STRAT, alpha=.10, lw=.7)
    ax[0, 0].plot(P["curves"][0], color=C_BASE, lw=2, label="Frozen parameters")
    ax[0, 0].plot(s["bh"].resample("1D").last(), color=C_BH, lw=1.2, label="Buy & hold")
    ax[0, 0].set_yscale("log"); ax[0, 0].set_title(f"{a}: {len(ok)} perturbed equity curves"); ax[0, 0].legend()

    ax[0, 1].hist(ok.total_return_pct, bins=30, color=C_STRAT, alpha=.8)
    ax[0, 1].axvline(base["total_return_pct"], color=C_BASE, lw=2, label="Baseline"); ax[0, 1].axvline(bh_ret, color=C_BH, lw=2, ls="--", label="Buy & hold")
    ax[0, 1].axvline(0, color="k", lw=.8); ax[0, 1].set_title("Total return %"); ax[0, 1].legend()

    ax[0, 2].hist(ok.sharpe, bins=30, color=C_STRAT, alpha=.8)
    ax[0, 2].axvline(base["sharpe"], color=C_BASE, lw=2, label="Baseline"); ax[0, 2].axvline(0, color="k", lw=.8)
    ax[0, 2].set_title("Sharpe ratio"); ax[0, 2].legend()

    ax[1, 0].hist(ok.max_dd_pct, bins=30, color=C_BAD, alpha=.75)
    ax[1, 0].axvline(base["max_dd_pct"], color=C_BASE, lw=2, label="Baseline"); ax[1, 0].set_title("Max drawdown %"); ax[1, 0].legend()

    sc = ax[1, 1].scatter(ok.max_dd_pct, ok.total_return_pct, c=ok.sharpe, cmap="viridis", s=22, alpha=.85)
    ax[1, 1].scatter(base["max_dd_pct"], base["total_return_pct"], color=C_BASE, s=130, marker="*", label="Baseline", zorder=5)
    ax[1, 1].set_xlabel("Max drawdown %"); ax[1, 1].set_ylabel("Total return %"); ax[1, 1].set_title("Return vs drawdown (colour = Sharpe)")
    plt.colorbar(sc, ax=ax[1, 1]); ax[1, 1].legend()

    xc = [c for c in ok.columns if c.startswith("x_") and ok[c].nunique() > 3]
    sens = pd.Series({c[2:]: spearmanr(ok[c], ok.sharpe)[0] for c in xc}).dropna()
    sens = sens.reindex(sens.abs().sort_values().index).tail(14)
    ax[1, 2].barh(sens.index, sens.values, color=np.where(sens.values > 0, "#2ca02c", C_BAD))
    ax[1, 2].axvline(0, color="k", lw=.8); ax[1, 2].set_title("Sensitivity: Spearman ρ (param multiplier vs Sharpe)")
    plt.tight_layout(); plt.show()

**Reading Part B**
* A robust strategy has a histogram that is tightly bunched around the baseline with a high `% runs profitable`. A baseline far to the right of the whole cloud means the frozen parameters were picked on a lucky peak.
* The sensitivity bars tell you *which* parameters matter: long bars = the result depends strongly on getting that parameter exactly right.

---
## Part C — Synthetic market test
Brand-new 15-min OHLCV paths (same schema as your CSVs, so the **unmodified** strategy code runs on them). Each path uses Student-t shocks (fat tails), stochastic volatility and OHLC wicks; volatility is scaled to each asset's own historical volatility.

| Scenario | What it simulates |
|---|---|
| `trend_bull` | persistent strong up-drift (~+90 %/yr log-drift) with ordinary pullbacks |
| `momentum_mild` | weak average drift but slowly-changing trends (positive autocorrelation of returns) |
| `bubble_burst` | accelerating parabolic run-up (typically ~10×) → blow-off top → crash with jump-downs (typically −80 %) → grinding bear |
| `chop_low_vol` | mean-reverting range, no drift, ~half normal volatility |

**Caveat:** the synthetic shocks are independent, so they contain none of the real-market microstructure (intraday momentum, volume-driven breakouts) that a breakout system may exploit. Treat the results as *regime stress-tests*, not as forecasts.

Paths with the same index use the same random shocks in all scenarios and both assets (common random numbers), so differences come from the regime, not luck.

In [ ]:
SCENARIOS = ["trend_bull", "momentum_mild", "bubble_burst", "chop_low_vol"]
BARS_PER_DAY = 96
DT = 1 / (365 * BARS_PER_DAY)

def scenario_profile(kind, n):
    x = np.linspace(0, 1, n)
    if kind == "trend_bull":
        return dict(mu=np.full(n, 0.90), vm=np.full(n, 0.90), drift_sd=0.9, drift_tau=60, vov=0.45)
    if kind == "momentum_mild":
        return dict(mu=np.full(n, 0.20), vm=np.full(n, 1.00), drift_sd=0.9, drift_tau=25, vov=0.45)
    if kind == "bubble_burst":
        mu = np.interp(x, [0, .45, .50, .51, .58, .62, 1.0], [0.5, 2.8, 0.3, -2.0, -2.0, -0.10, 0.30])
        vm = np.interp(x, [0, .45, .50, .51, .60, .70, 1.0], [0.8, 1.1, 1.3, 1.5, 1.3, 1.0, 0.7])
        crash = (x >= .51) & (x <= .58)
        return dict(mu=mu, vm=vm, drift_sd=0.4, drift_tau=20, vov=0.45,
                    jump_rate=np.where(crash, 5 / max(crash.sum(), 1), 0.0), jump_mu=-0.05, jump_sd=0.025)
    if kind == "chop_low_vol":
        return dict(mu=np.zeros(n), vm=np.full(n, 0.55), drift_sd=0.0, drift_tau=30, vov=0.15, revert_halflife_days=25)
    raise ValueError(kind)

def _ou(rng, n, tau_days, sd):
    if sd == 0: return np.zeros(n)
    a = np.exp(-1 / (tau_days * BARS_PER_DAY))
    return lfilter([1.0], [1.0, -a], rng.normal(0, sd * np.sqrt(1 - a * a), n))

def make_synthetic(kind, seed, p0, base_vol, med_volume, days=SYNTH_DAYS, start="2021-01-01"):
    rng = np.random.default_rng(seed); n = days * BARS_PER_DAY
    P = scenario_profile(kind, n)
    sig = base_vol * P["vm"] * np.exp(_ou(rng, n, 10, P["vov"]) - .5 * P["vov"] ** 2)
    df_t = 5; eps = rng.standard_t(df_t, n) * np.sqrt((df_t - 2) / df_t)
    sb = sig * np.sqrt(DT)
    if "revert_halflife_days" in P:
        a = np.exp(-np.log(2) / (P["revert_halflife_days"] * BARS_PER_DAY))
        logp = np.log(p0) + lfilter([1.0], [1.0, -a], sb * eps)
    else:
        mu = P["mu"] + _ou(rng, n, P["drift_tau"], P["drift_sd"])
        r = (mu - .5 * sig ** 2) * DT + sb * eps
        if "jump_rate" in P:
            jumps = rng.random(n) < P["jump_rate"]
            r = r + jumps * rng.normal(P["jump_mu"], P["jump_sd"], n)
        logp = np.log(p0) + np.cumsum(np.clip(r, -.4, .4))
    c = np.exp(logp); o = np.r_[p0, c[:-1]]
    wick = .48 * sb
    h = np.maximum(o, c) * np.exp(wick * np.abs(rng.normal(size=n)))
    l = np.minimum(o, c) * np.exp(-wick * np.abs(rng.normal(size=n)))
    vol = med_volume * np.exp(rng.normal(0, .5, n)) * (1 + np.abs(np.log(c / o)) / sb) / 1.8
    idx = pd.date_range(start, periods=n, freq="15min", tz="UTC", name="timestamp")
    d = pd.DataFrame(dict(open=o, high=h, low=l, close=c, volume=vol), index=idx)
    d["tradable"] = True
    return d

for a, s in SPEC.items():
    daily = s["d"].close.resample("1D").last()
    s["calib"] = dict(p0=float(s["d"].open.iloc[0]), base_vol=float(np.log(daily).diff().std() * np.sqrt(365)),
                      med_volume=float(s["d"].volume.median()))
    print(a, {k: round(v, 3) for k, v in s["calib"].items()})

In [ ]:
def path_summary(d):
    daily = d.close.resample("1D").last(); lr = np.log(daily).diff().dropna()
    return dict(total_return_pct=100 * (d.close.iloc[-1] / d.open.iloc[0] - 1),
                max_dd_pct=100 * (1 - d.close / d.close.cummax()).max(),
                ann_vol_pct=100 * lr.std() * np.sqrt(365), daily_kurtosis=lr.kurt(),
                hl_to_abs_ret=((d.high - d.low) / d.open).mean() / (d.close / d.open - 1).abs().mean())

fig, axes = plt.subplots(len(SPEC), 4, figsize=(19, 3.6 * len(SPEC)), squeeze=False)
rows = {}
for j, (a, s) in enumerate(SPEC.items()):
    real = s["d"]; rows[f"{a} REAL history"] = path_summary(real)
    for i, k in enumerate(SCENARIOS):
        d = make_synthetic(k, SEED, **s["calib"]); rows[f"{a} {k}"] = path_summary(d)
        ax = axes[j, i]; ax.plot(d.close.resample("1D").last(), color=C_STRAT, lw=1); ax.set_yscale("log")
        ax.set_title(f"{a} · {k}\nret {rows[f'{a} {k}']['total_return_pct']:+.0f}%  maxDD {rows[f'{a} {k}']['max_dd_pct']:.0f}%")
plt.tight_layout(); plt.show()
print("Sanity check – synthetic paths vs. the real history (vol, fat tails and candle-wick size should be in the same ballpark):")
display(pd.DataFrame(rows).T.round(2))

In [ ]:
SYN_ROWS, SYN_EX = [], {}
for a, s in SPEC.items():
    t0 = time.time()
    for k in SCENARIOS:
        for i in range(N_SYNTH_PATHS):
            d = make_synthetic(k, SEED + i, **s["calib"])
            bh = bh_curve(d, d.index); b = equity_metrics(bh)
            try:
                path, fills, tr, audit = s["mod"].run(d, s["cfg"], features=s["mod"].Features(d))
                m = equity_metrics(path.equity)
                row = dict(asset=a, scenario=k, path=i, **m, trades=len(tr), exposure_pct=100 * path.exposure.mean(),
                           bh_return_pct=b["total_return_pct"], bh_max_dd_pct=b["max_dd_pct"], bh_sharpe=b["sharpe"],
                           gross_breach=audit["gap_cap_breaches"] + audit["intrabar_cap_breaches"], failed=False)
                if i == 0: SYN_EX[(a, k)] = (d, path, tr, bh)
            except Exception as e:
                row = dict(asset=a, scenario=k, path=i, failed=True, error=str(e)[:80], bh_return_pct=b["total_return_pct"],
                           bh_max_dd_pct=b["max_dd_pct"], bh_sharpe=b["sharpe"])
            SYN_ROWS.append(row)
        print(f"  {a} · {k}: {N_SYNTH_PATHS} paths done  (cumulative {time.time()-t0:.0f}s)")
SYN = pd.DataFrame(SYN_ROWS)
SYN["excess_return_pct"] = SYN.total_return_pct - SYN.bh_return_pct
SYN.to_csv(OUT_DIR / "C_synthetic_runs.csv", index=False)
print("\nfailed runs:", int(SYN.failed.sum()), "| runs with >1x gross-exposure breach:", int((SYN.get("gross_breach", pd.Series(0)).fillna(0) > 0).sum()))

In [ ]:
for a, s in SPEC.items():
    fig, ax = plt.subplots(3, 4, figsize=(20, 11), sharex="col", gridspec_kw={"height_ratios": [2.2, 2, 1]})
    for j, k in enumerate(SCENARIOS):
        if (a, k) not in SYN_EX: continue
        d, path, tr, bh = SYN_EX[(a, k)]
        ax[0, j].plot(d.close, color="black", lw=.5); ax[0, j].set_yscale("log"); ax[0, j].set_title(f"{a} · {k}  (path 0)")
        for side, col, mk, nm in ((1, "tab:green", "^", "Long entry"), (-1, "tab:red", "v", "Short entry")):
            t = tr[tr.direction == side]
            if len(t): ax[0, j].scatter(t.entry_time, d.open.reindex(t.entry_time).values, s=22, color=col, marker=mk, label=nm, zorder=3)
        if j == 0: ax[0, j].legend(loc="upper left", fontsize=8)
        ax[1, j].plot(path.equity, color=C_STRAT, lw=1.2, label="Strategy"); ax[1, j].plot(bh, color=C_BH, lw=1, label="Buy & hold")
        ax[1, j].set_yscale("log")
        sm, bm = equity_metrics(path.equity), equity_metrics(bh)
        ax[1, j].set_title(f"Strategy {sm['total_return_pct']:+.0f}% (DD {sm['max_dd_pct']:.0f}%) vs B&H {bm['total_return_pct']:+.0f}% (DD {bm['max_dd_pct']:.0f}%)", fontsize=9)
        if j == 0: ax[1, j].legend(fontsize=8)
        ax[2, j].fill_between(path.index, path.signed_exposure, 0, step="post", color="tab:purple", alpha=.6); ax[2, j].set_ylim(-1.05, 1.05)
        if j == 0: ax[2, j].set_ylabel("Signed exposure")
    plt.tight_layout(); plt.show()

In [ ]:
ok = SYN[~SYN.failed]
for a in SPEC:
    sub = ok[ok.asset == a]
    fig, ax = plt.subplots(1, 3, figsize=(19, 4.8)); pos = np.arange(len(SCENARIOS))
    panels = [("Final equity multiple (x, log)", lambda d: 1 + d.total_return_pct / 100, lambda d: 1 + d.bh_return_pct / 100, True),
              ("Max drawdown %", lambda d: d.max_dd_pct, lambda d: d.bh_max_dd_pct, False),
              ("Sharpe ratio", lambda d: d.sharpe, lambda d: d.bh_sharpe, False)]
    for axi, (title, fs, fb, logy) in zip(ax, panels):
        data_s = [fs(sub[sub.scenario == k]).values for k in SCENARIOS]; data_b = [fb(sub[sub.scenario == k]).values for k in SCENARIOS]
        b1 = axi.boxplot(data_s, positions=pos - .18, widths=.3, patch_artist=True, showfliers=True)
        b2 = axi.boxplot(data_b, positions=pos + .18, widths=.3, patch_artist=True, showfliers=True)
        for bp, col in ((b1, C_STRAT), (b2, C_BH)):
            for p in bp["boxes"]: p.set_facecolor(col); p.set_alpha(.7)
            for m in bp["medians"]: m.set_color("black")
        if logy: axi.set_yscale("log"); axi.axhline(1, color="k", lw=.8, ls="--")
        else: axi.axhline(0, color="k", lw=.6)
        axi.set_xticks(pos); axi.set_xticklabels(SCENARIOS, rotation=12); axi.set_title(f"{a}: {title}")
    ax[0].legend([b1["boxes"][0], b2["boxes"][0]], ["Strategy", "Buy & hold"], loc="lower left")
    plt.tight_layout(); plt.show()

In [ ]:
g = ok.groupby(["asset", "scenario"])
score = pd.DataFrame({
    "paths": g.size(),
    "median strat return %": g.total_return_pct.median(), "median B&H return %": g.bh_return_pct.median(),
    "% paths profitable": g.apply(lambda x: 100 * (x.total_return_pct > 0).mean()),
    "% paths beating B&H": g.apply(lambda x: 100 * (x.total_return_pct > x.bh_return_pct).mean()),
    "5th pct strat return %": g.total_return_pct.quantile(.05),
    "median strat maxDD %": g.max_dd_pct.median(), "median B&H maxDD %": g.bh_max_dd_pct.median(),
    "95th pct strat maxDD %": g.max_dd_pct.quantile(.95),
    "median Sharpe": g.sharpe.median(), "median B&H Sharpe": g.bh_sharpe.median(),
    "median trades": g.trades.median(), "median exposure %": g.exposure_pct.median(),
}).reindex(pd.MultiIndex.from_product([list(SPEC), SCENARIOS], names=["asset", "scenario"]))
display(score.round(1))
score.to_csv(OUT_DIR / "C_synthetic_scorecard.csv")

---
## Final — one-page scorecard

In [ ]:
final = {}
for a, s in SPEC.items():
    P = PARAM[a]["summary"]; t = tables[a]
    row = {"historical return %": baseline.loc[f"{a} strategy", "total_return_pct"],
           "historical max DD %": baseline.loc[f"{a} strategy", "max_dd_pct"],
           "A · bootstrap P(loss) %": t.loc["Bootstrap", "prob_loss"],
           "A · bootstrap 5th-pct return %": t.loc["Bootstrap", "p5_ret"],
           "A · shuffle 95th-pct DD %": t.loc["Shuffle", "p95_dd"],
           "B · % perturbed runs profitable": P["% runs profitable"],
           "B · median perturbed Sharpe / baseline": P["median Sharpe"] / P["baseline Sharpe"] if P["baseline Sharpe"] else np.nan,
           "B · 95th-pct perturbed DD %": P["95th pct max DD %"]}
    for k in SCENARIOS:
        row[f"C · {k}: % paths profitable"] = score.loc[(a, k), "% paths profitable"]
        row[f"C · {k}: median ret %"] = score.loc[(a, k), "median strat return %"]
        row[f"C · {k}: median maxDD %"] = score.loc[(a, k), "median strat maxDD %"]
    final[a] = row
final = pd.DataFrame(final)
display(final.round(1))
final.to_csv(OUT_DIR / "final_scorecard.csv")
print(f"All tables were also saved as CSV in: {OUT_DIR.resolve()}")